# VASP dipole correction and bounded gradient descent

Prepare the two special VASP branches validated remotely:

- ZnO/GaN single interface with dipole correction.
- LiCoO₂ double interface with VASP gradient descent and a 0.2 Å maximum rigid-translation step.

The GD example deliberately uses one iteration as a workflow smoke test.

In [ ]:
from pathlib import Path
import json
import sys

import InterOptimus

candidates = (Path.cwd(), Path.cwd().parent, Path(InterOptimus.__file__).resolve().parents[1])
REPO_ROOT = next((path for path in candidates if (path / "scripts").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the InterOptimus repository or its examples directory")
sys.path.insert(0, str(REPO_ROOT))

from InterOptimus.agents.simple_iomaker import run_simple_iomaker
from scripts.prepare_interface_workflow_validation import structures, workflow

WORKDIR = Path.cwd() / "remote_vasp_special_examples"
SUBMIT = False
VASP_PARTITION = "spr"
WORKDIR.mkdir(exist_ok=True)
(WORKDIR / "materials").mkdir(exist_ok=True)
(WORKDIR / "configs").mkdir(exist_ok=True)

materials = structures()
for name in ("zno", "gan", "lco_a", "lco_b"):
    materials[name].to(filename=WORKDIR / "materials" / f"{name}.cif")

configs = [
    workflow(name="example_vasp_dipole_zno_gan", film="zno", substrate="gan", calc="sevenn", worker="sevenn", miller=(0, 0, 1), max_area=15, do_vasp=True, dipole=True, pair_selection="each_plane_lowest"),
    workflow(name="example_vasp_gd_lco", film="lco_a", substrate="lco_b", calc="dpa", worker="dpa", miller=(0, 0, 1), max_area=12, double_interface=True, do_vasp=True, do_vasp_gd=True),
]
for config in configs:
    bulk = config["IO_workflow_config"]["bulk_cifs"]
    bulk["film_cif"] = str((WORKDIR / bulk["film_cif"]).resolve())
    bulk["substrate_cif"] = str((WORKDIR / bulk["substrate_cif"]).resolve())
    config["cluster"]["vasp"]["vasp_slurm_partition"] = VASP_PARTITION
    path = WORKDIR / "configs" / f"{config['workflow_name']}.json"
    path.write_text(json.dumps(config, indent=2))

configs[1]["IO_workflow_config"]["vasp_settings"]["vasp_gd_kwargs"]

In [ ]:
results = {}
if SUBMIT:
    for config in configs:
        result = run_simple_iomaker(config)
        results[config["workflow_name"]] = result
        print(config["workflow_name"], result["mlip_job_uuid"])
else:
    print("Configuration only. Confirm the 0.2 Å GD bound and cluster settings before submission.")

results